# Bronze Layer — Yelp Users (Kafka → Iceberg)

Đọc stream từ topic `raw_yelp_users`, ghi raw vào `nessie.bronze.yelp_users`.
Bronze giữ nguyên tất cả records kể cả duplicate — đây là raw landing zone.

In [1]:
from pyspark.sql import SparkSession

try:
    spark.stop()
except:
    pass

spark = SparkSession.builder \
    .appName("Yelp_Bronze_Users") \
    .config("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog") \
    .config("spark.sql.catalog.nessie.uri", "http://nessie:19120/api/v1") \
    .config("spark.sql.catalog.nessie.ref", "main") \
    .config("spark.sql.catalog.nessie.warehouse", "s3a://warehouse/") \
    .config("spark.sql.catalog.nessie.s3.endpoint", "http://minio:9000") \
    .config("spark.sql.catalog.nessie.io-impl", "org.apache.iceberg.io.ResolvingFileIO") \
    .config("spark.sql.catalog.nessie.s3.path-style-access", "true") \
    .config("spark.sql.catalog.nessie.s3.access-key-id", "admin") \
    .config("spark.sql.catalog.nessie.s3.secret-access-key", "password") \
    .config("spark.sql.defaultCatalog", "nessie") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .getOrCreate()

# Fix S3A credentials sau getOrCreate()
hc = spark.sparkContext._jsc.hadoopConfiguration()
hc.set("fs.s3a.endpoint",          "http://minio:9000")
hc.set("fs.s3a.access.key",        "admin")
hc.set("fs.s3a.secret.key",        "password")
hc.set("fs.s3a.path.style.access", "true")
hc.set("fs.s3a.connection.ssl.enabled", "false")
hc.set("fs.s3a.impl",              "org.apache.hadoop.fs.s3a.S3AFileSystem")
hc.set("fs.s3a.aws.credentials.provider",
       "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")

spark.sparkContext.setLogLevel("ERROR")
print(f"✅ SparkSession ready | Endpoint: {hc.get('fs.s3a.endpoint')}")

✅ SparkSession ready | Endpoint: http://minio:9000


26/06/16 01:15:15 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
!pip install boto3


[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [3]:
import boto3
from botocore.client import Config

s3 = boto3.client(
    "s3", endpoint_url="http://minio:9000",
    aws_access_key_id="admin", aws_secret_access_key="password",
    config=Config(signature_version="s3v4")
)

buckets = [b["Name"] for b in s3.list_buckets()["Buckets"]]
if "warehouse" not in buckets:
    s3.create_bucket(Bucket="warehouse")
    print("✅ Tạo bucket warehouse")
else:
    print("✅ Bucket warehouse đã tồn tại")

✅ Bucket warehouse đã tồn tại


In [4]:
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType
from pyspark.sql.functions import from_json, col

# Stop bất kỳ stream cũ nào
for q in spark.streams.active:
    q.stop()
    print(f"⏹ Stopped stream: {q.name}")

# Schema phải khớp với SCD_FIELDS trong Kafka producer
schema = StructType([
    StructField("user_id",       StringType(), True),
    StructField("name",          StringType(), True),
    StructField("review_count",  LongType(),   True),
    StructField("yelping_since", StringType(), True),
    StructField("useful",        LongType(),   True),
    StructField("funny",         LongType(),   True),
    StructField("cool",          LongType(),   True),
    StructField("fans",          LongType(),   True),
    StructField("average_stars", DoubleType(), True),
    StructField("elite",         StringType(), True),
])

# 1. Tạo namespace bronze
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.bronze")

# 2. Tạo bảng Bronze (nếu chưa có)
spark.sql("""
    CREATE TABLE IF NOT EXISTS nessie.bronze.yelp_users (
        user_id       STRING,
        name          STRING,
        review_count  BIGINT,
        yelping_since STRING,
        useful        BIGINT,
        funny         BIGINT,
        cool          BIGINT,
        fans          BIGINT,
        average_stars DOUBLE,
        elite         STRING
    ) USING iceberg
""")

print("✅ Table nessie.bronze.yelp_users ready")

✅ Table nessie.bronze.yelp_users ready


In [5]:
# 3. Đọc stream từ Kafka
kafka_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "raw_yelp_users") \
    .option("startingOffsets", "earliest") \
    .option("failOnDataLoss", "false") \
    .option("maxOffsetsPerTrigger", 100000) \
    .load()

# 4. Parse JSON
parsed = kafka_stream \
    .selectExpr("CAST(value AS STRING) as json_str") \
    .select(from_json(col("json_str"), schema).alias("data")) \
    .select("data.*") \
    .filter(col("user_id").isNotNull())

# 5. Ghi vào Bronze Iceberg
print("[*] Khởi chạy Bronze streaming...")

bronze_query = parsed.writeStream \
    .format("iceberg") \
    .outputMode("append") \
    .option("checkpointLocation", "s3a://warehouse/checkpoints/bronze_yelp_users") \
    .trigger(processingTime="15 seconds") \
    .queryName("bronze_yelp_users") \
    .toTable("nessie.bronze.yelp_users")

print("[+] Bronze stream đang chạy! Topic: raw_yelp_users → nessie.bronze.yelp_users")

[*] Khởi chạy Bronze streaming...
[+] Bronze stream đang chạy! Topic: raw_yelp_users → nessie.bronze.yelp_users


In [10]:
# Monitor số lượng bản ghi trong Bronze và trạng thái stream mỗi 15 giây
import time

for i in range(1):
    try:
        # Chỉ lấy snapshot mới nhất, không cộng dồn lịch sử cũ
        total = spark.sql("""
            SELECT CAST(summary['total-records'] AS LONG) AS total
            FROM nessie.bronze.yelp_users.snapshots
            ORDER BY committed_at DESC
            LIMIT 1
        """).collect()[0]["total"]
        active = len(spark.streams.active)
        print(f"[{i+1:02d}] Bronze records: {total:>10,} | Streams active: {active}")
    except Exception as e:
        print(f"[{i+1:02d}] Error: {e}")
    time.sleep(15)


[01] Bronze records:  2,783,056 | Streams active: 1
